# 03_gold_aggregation


Aggregate retail order data from the Silver layer into a Gold Delta table containing revenue, order counts, and units sold by store and month.

## Table of Contents
1. Import PySpark and load Silver data
2. Calculate revenue after discounts
3. Aggregate revenue by store and month
4. Create Gold schema and save the Delta table
5. Validate the Gold table

### Import PySpark and load Silver data

In [0]:
df = spark.table("dataengineering_silver.cleaned_data.butiksordrer")

df.printSchema()
display(df.limit(10))

### Calculate revenue after discounts

In [0]:
from pyspark.sql.functions import col, round

df = df.withColumn(
    "omsaetning",
    round(
        col("antal") * col("stykpris") * (1 - col("rabat_pct") / 100),
        2
        )
)

display(df.limit(10))

### Aggregate revenue by store and month

In [0]:
from pyspark.sql import functions as F

df_gold = (
    df
    .withColumn("maaned", F.date_format("ordre_dato", "yyyy-MM"))
    .groupBy("butik", "maaned")
    .agg(
        F.round(F.sum("omsaetning"), 2).alias("samlet_omsaetning"),
        F.countDistinct("ordre_id").alias("antal_ordrer"),
        F.sum("antal").alias("antal_solgte_enheder")
    )
    .orderBy("butik", "maaned")
)

display(df_gold)

### Create Gold schema and save the Delta table

In [0]:
# 4. Opret Gold-schema, hvis det ikke findes
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    dataenginerring_gold.aggregated_data
""")

# Gem Gold-tabellen
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "dataenginerring_gold.aggregated_data.omsaetning_pr_butik_maaned"
    )

### Validate the Gold table

In [0]:
# 5. Kontrollér den gemte Gold-tabel
df_check = spark.table(
    "dataenginerring_gold.aggregated_data.omsaetning_pr_butik_maaned"
)

display(df_check)